In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType
)

# Streaming storage paths
BASE_PATH = "/Volumes/workspace/default/olist_raw"

INPUT_PATH = f"{BASE_PATH}/stream_input"
CHECKPOINT_PATH = f"{BASE_PATH}/stream_checkpoint"

# Output Delta table
OUTPUT_TABLE = "workspace.bronze.order_events_stream"

# Create directories only if they don't already exist
dbutils.fs.mkdirs(INPUT_PATH)
dbutils.fs.mkdirs(CHECKPOINT_PATH)

print("Streaming directories ready!")

# Load Silver orders
orders = spark.table("workspace.silver.orders")

print("Streaming environment initialized!")
print("Source records:", orders.count())
print("Input path:", INPUT_PATH)

Streaming directories ready!
Streaming environment initialized!
Source records: 99441
Input path: /Volumes/workspace/default/olist_raw/stream_input


In [0]:

from pyspark.sql.window import Window

# Check whether historical event files already exist
existing_files = dbutils.fs.ls(INPUT_PATH)

event_files_exist = any(
    file.name.startswith("part-")
    for file in existing_files
)

if event_files_exist:
    print("Historical event files already exist.")
    print("Skipping event generation to prevent duplicates.")

else:
    # Prepare historical order events
    event_source = (
        orders
        .filter(F.col("order_purchase_timestamp").isNotNull())
        .select(
            "order_id",
            "customer_id",
            "order_status",
            F.col("order_purchase_timestamp")
             .cast("string")
             .alias("event_timestamp")
        )
    )

    window_spec = Window.orderBy(
        "event_timestamp",
        "order_id"
    )

    event_source = event_source.withColumn(
        "event_sequence",
        F.row_number().over(window_spec)
    )

    # Generate five batches
    for batch_number in range(5):

        start = batch_number * 2000 + 1
        end = start + 1999

        batch_df = (
            event_source
            .filter(
                (F.col("event_sequence") >= start) &
                (F.col("event_sequence") <= end)
            )
            .drop("event_sequence")
        )

        batch_count = batch_df.count()

        if batch_count > 0:
            (
                batch_df
                .coalesce(1)
                .write
                .mode("append")
                .json(INPUT_PATH)
            )

        print(
            f"Batch {batch_number + 1}: "
            f"{batch_count} events generated"
        )

    print("Historical event generation completed!")


Historical event files already exist.
Skipping event generation to prevent duplicates.


In [0]:
# Read generated JSON events as a batch
generated_events = spark.read.json(INPUT_PATH)

print("Total generated events:", generated_events.count())

display(
    generated_events.orderBy("event_timestamp").limit(10)
)

Total generated events: 10000


customer_id,event_timestamp,order_id,order_status
08c5351a6aca1c1589a38f244edeee9d,2016-09-04 21:15:19,2e7a8482f6fb09756ca50c10d7bfc047,shipped
683c54fc24d40ee9f8a6fc179fd9856c,2016-09-05 00:15:34,e5fa5a7210941f7d56d0208e4e071d35,canceled
622e13439d6b5a0b486c435618b2679e,2016-09-13 15:24:19,809a282bbd5dbcabb6f2f724fca862ec,canceled
86dc2ffce2dfff336de2f386a786e574,2016-09-15 12:16:38,bfbd0f9bdef84302105ad712db648a6c,delivered
b106b360fe2ef8849fbbd056f777b4d5,2016-10-02 22:07:52,71303d7e93b399f5bcd537d124c0bcfa,canceled
355077684019f7f60a031656bd7262b8,2016-10-03 09:44:50,3b697a20d9e427646d92567910af6d57,delivered
7ec40b22510fdbea1b08921dd39e63d8,2016-10-03 16:56:50,be5bc2f0da14d8071e2d45451ad119d9,delivered
70fc57eeae292675927697fe03ad3ff5,2016-10-03 21:01:41,65d1e226dfaeb8cdc42f665422522d14,canceled
6f989332712d3222b6571b1cf5b835ce,2016-10-03 21:13:36,a41c8759fbe7aab36ea07e038b2d4465,delivered
b8cf418e97ae795672d326288dfab7a7,2016-10-03 22:06:03,d207cc272675637bfed0062edffd0818,delivered


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType
)
from pyspark.sql import functions as F

# Define incoming event schema
event_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("event_timestamp", StringType(), True)
])

# Create streaming DataFrame
streaming_df = (
    spark.readStream
    .format("json")
    .schema(event_schema)
    .load(INPUT_PATH)
)

print("Streaming source initialized!")
print("Is streaming:", streaming_df.isStreaming)

Streaming source initialized!
Is streaming: True


In [0]:
processed_stream = (
    streaming_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "event_date",
        F.to_date("event_timestamp")
    )
    .filter(
        F.col("order_id").isNotNull()
    )
)

print("Streaming transformations configured!")

Streaming transformations configured!


In [0]:
OUTPUT_TABLE = "workspace.bronze.order_events_stream"

streaming_query = (
    processed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(OUTPUT_TABLE)
)

print("Streaming execution completed!")

Streaming execution completed!


In [0]:
streamed_orders = spark.table(
    "workspace.bronze.order_events_stream"
)

print(
    "Total streamed events:",
    streamed_orders.count()
)

print(
    "Unique orders:",
    streamed_orders.select("order_id").distinct().count()
)

display(
    streamed_orders.orderBy("event_timestamp").limit(10)
)

Total streamed events: 10000
Unique orders: 10000


order_id,customer_id,order_status,event_timestamp,processed_timestamp,event_date
2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04T21:15:19.000Z,2026-11-18T18:06:46.913Z,2016-09-04
e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05T00:15:34.000Z,2026-11-18T18:06:46.913Z,2016-09-05
809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2016-09-13T15:24:19.000Z,2026-11-18T18:06:46.913Z,2016-09-13
bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15T12:16:38.000Z,2026-11-18T18:06:46.913Z,2016-09-15
71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,canceled,2016-10-02T22:07:52.000Z,2026-11-18T18:06:46.913Z,2016-10-02
3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03T09:44:50.000Z,2026-11-18T18:06:46.913Z,2016-10-03
be5bc2f0da14d8071e2d45451ad119d9,7ec40b22510fdbea1b08921dd39e63d8,delivered,2016-10-03T16:56:50.000Z,2026-11-18T18:06:46.913Z,2016-10-03
65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03T21:01:41.000Z,2026-11-18T18:06:46.913Z,2016-10-03
a41c8759fbe7aab36ea07e038b2d4465,6f989332712d3222b6571b1cf5b835ce,delivered,2016-10-03T21:13:36.000Z,2026-11-18T18:06:46.913Z,2016-10-03
d207cc272675637bfed0062edffd0818,b8cf418e97ae795672d326288dfab7a7,delivered,2016-10-03T22:06:03.000Z,2026-11-18T18:06:46.913Z,2016-10-03
